In [2]:
import pandas as pd
import numpy as np
import re
import spacy
import textstat
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split

# Tải công cụ NLP
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"]) # Tắt ner/parser để chạy nhanh hơn
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("--- Đang tải dữ liệu từ Hugging Face ---")
dataset = load_dataset("chillies/IELTS-writing-task-2-evaluation")
df = pd.concat([pd.DataFrame(dataset['train']), pd.DataFrame(dataset['test'])], ignore_index=True)
print(f"Tổng số mẫu ban đầu: {len(df)}")

C:\Users\WINDOWS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WINDOWS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


--- Đang tải dữ liệu từ Hugging Face ---
Tổng số mẫu ban đầu: 10324


### 1. Data cleaning

In [3]:
# 1. Chuyển đổi Band Score sang số và làm tròn 0.5
df['band'] = pd.to_numeric(df['band'], errors='coerce')
df = df.dropna(subset=['band'])
df['band'] = (df['band'] * 2).round() / 2

# 2. Chuẩn hóa văn bản
def basic_clean(text):
    if not isinstance(text, str): return ""
    text = text.replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', text).strip()

df['essay'] = df['essay'].apply(basic_clean)
df['prompt'] = df['prompt'].apply(basic_clean)

# 3. Xóa trùng lặp & Lọc bài quá ngắn (Dưới 150 từ không đủ chất lượng IELTS)
df = df.drop_duplicates(subset=['essay']).reset_index(drop=True)
df['essay_len'] = df['essay'].apply(lambda x: len(x.split()))
df = df[df['essay_len'] >= 150].reset_index(drop=True)

print(f"Số lượng sau khi làm sạch & lọc trùng: {len(df)}")

Số lượng sau khi làm sạch & lọc trùng: 8273


### 2. Trích xuất điểm thành phần (Regex)

In [4]:
def extract_sub_scores(text):
    patterns = {
        'TR': r'(?:Task (?:Achievement|Response)):?\s*(?:\[|:)?\s*(\d+(?:\.\d+)?)',
        'CC': r'(?:Cohesion (?:and|&)? Coherence|Coherence (?:and|&)? Cohesion):?\s*(?:\[|:)?\s*(\d+(?:\.\d+)?)',
        'LR': r'(?:Lexical Resource|Vocabulary):?\s*(?:\[|:)?\s*(\d+(?:\.\d+)?)',
        'GRA': r'(?:Grammar[^:]*|Grammatical Range and Accuracy):?\s*(?:\[|:)?\s*(\d+(?:\.\d+)?)'
    }
    res = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        res[key] = float(match.group(1)) if match else np.nan
    return pd.Series(res)

print("--- Đang trích xuất điểm TR, CC, LR, GRA ---")
df[['TR', 'CC', 'LR', 'GRA']] = df['evaluation'].apply(extract_sub_scores)

# Điền các điểm thành phần bị thiếu bằng điểm Band tổng (giả định logic)
for col in ['TR', 'CC', 'LR', 'GRA']:
    df[col] = df[col].fillna(df['band'])

--- Đang trích xuất điểm TR, CC, LR, GRA ---


### 3. Feature Engineering nâng cao (Linguistic & Semantic)

In [5]:
def get_advanced_features(df):
    # 1. Tính Semantic Relevance giữa Prompt và Essay (S-BERT)
    print("Tính toán Prompt-Essay Relevance...")
    prompt_embeddings = sbert_model.encode(df['prompt'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    essay_embeddings = sbert_model.encode(df['essay'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    cosine_scores = util.cos_sim(prompt_embeddings, essay_embeddings)
    df['prompt_relevance'] = cosine_scores.diag().cpu().numpy()

    # 2. Tính Lexical Diversity & Readability
    print("Tính toán Linguistic Features...")
    ttr_scores = []
    readability_scores = []

    for doc in tqdm(nlp.pipe(df['essay'], batch_size=50), total=len(df)):
        # Lexical Diversity (Type-Token Ratio)
        tokens = [t.text.lower() for t in doc if not t.is_punct and not t.is_space]
        ttr = len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0
        ttr_scores.append(ttr)
        
        # Readability Score
        readability_scores.append(textstat.flesch_reading_ease(doc.text))

    df['lexical_diversity'] = ttr_scores
    df['readability_score'] = readability_scores
    return df

df = get_advanced_features(df)

Tính toán Prompt-Essay Relevance...


Batches: 100%|██████████| 259/259 [04:34<00:00,  1.06s/it]


Tính toán Linguistic Features...


100%|██████████| 8273/8273 [05:01<00:00, 27.43it/s]


### 4. Tạo Rationale & Prompt Template để Fine-tuning LLM

In [6]:
def create_instruction(row):
    # Tạo một đoạn mô tả các đặc trưng đã trích xuất được
    rationale = f"This essay has a lexical diversity of {row['lexical_diversity']:.2f}, " \
                f"a readability score of {row['readability_score']:.1f}, " \
                f"and its relevance to the prompt is {row['prompt_relevance']:.2f}."
    
    # Cấu trúc hóa output mong muốn cho Model
    # Model sẽ học cách sinh ra rationale -> điểm thành phần -> điểm tổng
    target_output = f"Analysis: {rationale} Detailed Evaluation: {row['evaluation']} " \
                    f"Scores: TR: {row['TR']}, CC: {row['CC']}, LR: {row['LR']}, GRA: {row['GRA']} " \
                    f"Final Band: {row['band']}"
    return target_output

df['target_text'] = df.apply(create_instruction, axis=1)

### 5. Chia tập data

In [7]:
# Sử dụng Stratified Split để đảm bảo cân bằng điểm số ở các tập
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['band'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['band'])

print(f"\nKích thước tập Train: {len(train_df)}")
print(f"Kích thước tập Val: {len(val_df)}")
print(f"Kích thước tập Test: {len(test_df)}")

# Lưu ra các file CSV
train_df.to_csv("ielts_train_df.csv", index=False)
val_df.to_csv("ielts_val_df.csv", index=False)
test_df.to_csv("ielts_test_df.csv", index=False)


Kích thước tập Train: 6618
Kích thước tập Val: 827
Kích thước tập Test: 828
